# Kyiv commuter alert forecaster — train in Google Colab

Two models, every 15 minutes, up to 3 hours ahead:
* **onset** — no alert now: chance a new Kyiv alert starts within 15–180 min (Livoberezhna scenario);
* **clear** — alert running: chance of the all-clear within 15–180 min (Arsenalna scenario).

Everything needed is in the repository: cleaned Kyiv alert labels (`data/kyiv/`), text signals from four Telegram
channels (`data/kyiv/signals.csv.gz`), and the regional alert history (downloaded by `scripts/get_data.sh`).
Runtime: about 1 h on a free CPU runtime (no GPU needed).

## Forecast right now (no training needed)
Run the first cell below, then `!python kyiv/live.py`. Training cells follow.

In [ ]:
# 1. Get the code and data (replace with your repository URL, or upload the zip and unzip it)
!git clone https://github.com/MrVenitar/Kyiv-air-raid-forecast.git
%cd Kyiv-air-raid-forecast
!pip -q install -r requirements.txt
!bash scripts/get_data.sh            # regional alert history, pinned snapshot

In [ ]:
# 2. Leakage tests: features at any minute must equal a rebuild from data published before it
!python kyiv/test_kyiv_leakage.py

In [ ]:
# 3. 15-minute tables (onset + all-clear), ~1 min
!python kyiv/train.py --stage tables

In [ ]:
# 4. Validation: model, recency weighting, recalibration and feature groups are chosen here (~40 min)
!python kyiv/train.py --stage validate --task onset
!python kyiv/train.py --stage validate --task clear

In [ ]:
# 5. Test: run once with the frozen choices (~15 min), then fit the final models
!python kyiv/train.py --stage test
!python kyiv/train.py --stage final
!cat reports/kyiv/test.md

In [ ]:
# 6. Forecast at any moment covered by the data (Kyiv local time)
!python kyiv/forecast.py --at "2026-10-10 08:15"
!python kyiv/forecast.py

In [ ]:
# Live forecast for the current minute (reads the newest posts from public Telegram pages)
!python kyiv/live.py